# 🤖 Budujemy agenta AI w Pythonie

Krok po kroku zbudujemy agenta, ktory:
1. Wywoluje model LLM
2. Decyduje, ktore **narzedzie** uzyc
3. Pamięta kontekst rozmowy
4. Zwraca **strukturalny JSON** (nie wolny tekst)
5. Pracuje na **prawdziwych danych** (large_tickets.csv)

**Czas:** ~60 minut | **Wymagania:** Google Colab z GPU T4

## 🛠️ 0. Instalacja i konfiguracja

Instalujemy Ollama na Colab i pobieramy model. Tak — Ollama dziala na Colab!

### 📖 Wytlumaczenie:
Ollama uruchamia modele lokalnie przez API REST. Na Colab instalujemy go jak na kazdym Linuxie, a potem komunikujemy sie przez `http://localhost:11434`.

### 💡 Cwiczenie:
Po uruchomieniu sprawdz dostepne modele: `!ollama list`

In [ ]:
# Install Ollama on Colab
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in background
import subprocess
import time

process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)  # wait for server to start

# Pull a small model
!ollama pull gemma3:4b

print("\n✅ Ollama gotowe!")
!ollama list

In [ ]:
import requests
import json
import pandas as pd

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "gemma3:4b"

# Fallback: if Ollama is not available, use pre-computed responses
USE_OLLAMA = True
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=3)
    print(f"✅ Ollama dziala! Modele: {[m['name'] for m in r.json().get('models', [])]}")
except:
    USE_OLLAMA = False
    print("⚠️ Ollama niedostepne — uzywam trybu fallback")

---
## 🔹 Krok 1: Podstawowe wywolanie LLM (~10 min)

Zaczynamy od najprostszej rzeczy: wysylamy pytanie do modelu i dostajemy odpowiedz.

### 📖 Wytlumaczenie:
Kazdy agent AI zaczyna od jednej umiejetnosci: wywolania modelu. To jest nasz fundament. Funkcja `call_llm()` to "mozg" agenta.

### 💡 Cwiczenie:
Zmien `system_prompt` na "Odpowiadaj tylko po polsku" i sprawdz efekt.

In [ ]:
def call_llm(prompt, system_prompt="You are a helpful assistant."):
    """Call the LLM and return the response text."""
    if not USE_OLLAMA:
        return f"[FALLBACK] Response to: {prompt[:50]}..."

    response = requests.post(OLLAMA_URL, json={
        "model": MODEL,
        "prompt": prompt,
        "system": system_prompt,
        "stream": False,
    }, timeout=60)

    return response.json()["response"].strip()

# Test it!
answer = call_llm("What is machine learning? Answer in 2 sentences.")
print(f"🤖 Model odpowiedzial:\n{answer}")

In [ ]:
# Test with Polish
answer_pl = call_llm(
    "Czym jest sztuczna inteligencja? Odpowiedz w 2 zdaniach.",
    system_prompt="Odpowiadaj zawsze po polsku, krotko i zrozumiale."
)
print(f"🤖 Po polsku:\n{answer_pl}")

---
## 🔹 Krok 2: Narzedzia (Tools) — agent decyduje co zrobic (~15 min)

Agent to nie chatbot. Agent ma **narzedzia** i sam decyduje, ktorego uzyc.

### 📖 Wytlumaczenie:
Definiujemy 3 funkcje Pythona (narzedzia). Wysylamy modelowi opis narzedzi + pytanie uzytkownika. Model odpowiada, ktore narzedzie wywolac i z jakimi parametrami. Kod Pythona faktycznie je wywoluje. To jest serce kazdego agenta AI w 2026.

### 💡 Cwiczenie:
Dodaj 4. narzedzie: `get_team_email` ktore zwraca adres e-mail zespolu na podstawie kategorii.

In [ ]:
# === Define tools ===

def classify_ticket(text):
    """Classify an IT ticket into a category."""
    prompt = (
        "Classify this IT support ticket into exactly one category: "
        "Hardware, Network, Software, Account.\n\n"
        f"Ticket: \"{text}\"\n\n"
        "Reply with ONLY the category name, nothing else."
    )
    return call_llm(prompt)


def search_knowledge_base(query):
    """Search the IT knowledge base for relevant information."""
    kb = {
        "printer": "Network printers (IP/WiFi) → classify as Network. Physical printer issues → Hardware.",
        "wifi": "WiFi login problems are Network issues, not Account issues.",
        "password": "Password changes in any app (ERP, CRM) → classify as Account.",
        "access": "Access issues after password change → Account (AD sync needed).",
        "vpn": "VPN issues → Network. Check certificate expiry and firewall.",
        "antivirus": "Antivirus blocking software → Software. Add exception.",
    }
    query_lower = query.lower()
    results = [v for k, v in kb.items() if k in query_lower]
    return " | ".join(results) if results else "No relevant articles found."


def get_ticket_status(ticket_id):
    """Check the status of a ticket by ID."""
    # Simulated ticket database
    statuses = {
        "T-001": {"status": "Open", "assigned_to": "Network Team", "priority": "P2"},
        "T-002": {"status": "In Progress", "assigned_to": "Hardware Team", "priority": "P1"},
        "T-003": {"status": "Resolved", "assigned_to": "Account Team", "priority": "P3"},
    }
    return json.dumps(statuses.get(ticket_id, {"error": f"Ticket {ticket_id} not found"}), indent=2)


# Tool registry
TOOLS = {
    "classify_ticket": classify_ticket,
    "search_knowledge_base": search_knowledge_base,
    "get_ticket_status": get_ticket_status,
}

TOOL_DESCRIPTIONS = """
Available tools:
1. classify_ticket(text) — classify an IT ticket into Hardware/Network/Software/Account
2. search_knowledge_base(query) — search internal IT docs for troubleshooting info
3. get_ticket_status(ticket_id) — check status of existing ticket (e.g. T-001)
"""

print("✅ 3 narzedzia zdefiniowane:")
for name, func in TOOLS.items():
    print(f"  🔧 {name}: {func.__doc__}")

In [ ]:
# === Agent: decide which tool to use ===

AGENT_SYSTEM_PROMPT = f"""You are an IT helpdesk agent. You have access to tools.
{TOOL_DESCRIPTIONS}
When the user asks something, decide which tool to use.
Reply in this EXACT JSON format (nothing else):
{{"tool": "tool_name", "argument": "the argument to pass"}}

If no tool is needed, reply:
{{"tool": "none", "argument": "your direct answer"}}"""


def agent_decide(user_input):
    """Agent decides which tool to call based on user input."""
    response = call_llm(user_input, system_prompt=AGENT_SYSTEM_PROMPT)

    # Parse JSON from response
    try:
        # Find JSON in response
        start = response.index("{")
        end = response.rindex("}") + 1
        decision = json.loads(response[start:end])
        return decision
    except (ValueError, json.JSONDecodeError):
        return {"tool": "none", "argument": response}


def agent_execute(user_input):
    """Full agent loop: decide → execute → respond."""
    print(f"👤 Uzytkownik: {user_input}")

    # Step 1: Agent decides
    decision = agent_decide(user_input)
    tool_name = decision.get("tool", "none")
    argument = decision.get("argument", "")
    print(f"🧠 Agent decyduje: tool={tool_name}, argument={argument}")

    # Step 2: Execute tool
    if tool_name in TOOLS:
        result = TOOLS[tool_name](argument)
        print(f"🔧 Wynik narzedzia: {result}")
        return result
    else:
        print(f"💬 Odpowiedz bezposrednia: {argument}")
        return argument


# Test the agent!
print("=" * 60)
agent_execute("Classify this ticket: Network printer is not responding")
print("\n" + "=" * 60)
agent_execute("Search the knowledge base for printer issues")
print("\n" + "=" * 60)
agent_execute("What is the status of ticket T-001?")

---
## 🔹 Krok 3: Pamiec — agent zapamietuje kontekst (~10 min)

Dotychczas kazde pytanie bylo niezalezne. Teraz dodajemy pamiec — agent pamięta, o czym rozmawialismy.

### 📖 Wytlumaczenie:
Pamiec to lista poprzednich wiadomosci. Przy kazdym wywolaniu dodajemy cala historie do prompta. Model widzi kontekst i moze odpowiadac na pytania typu: "A jaki byl wynik?" bez powtarzania.

### 💡 Cwiczenie:
Dodaj limit pamieci: agent pamięta tylko ostatnie 5 wiadomosci. Dlaczego to wazne? (context window!)

In [ ]:
# === Agent with memory ===

class ITAgent:
    def __init__(self):
        self.memory = []
        self.system_prompt = AGENT_SYSTEM_PROMPT

    def chat(self, user_input):
        """Process user input with memory context."""
        # Build context from memory
        context = ""
        if self.memory:
            context = "Previous conversation:\n"
            for msg in self.memory[-5:]:  # last 5 exchanges
                context += f"User: {msg['user']}\nAgent: {msg['response']}\n"
            context += "\n"

        # Agent decides and executes
        full_prompt = context + f"User: {user_input}"
        decision = agent_decide(full_prompt)

        tool_name = decision.get("tool", "none")
        argument = decision.get("argument", "")

        if tool_name in TOOLS:
            result = TOOLS[tool_name](argument)
        else:
            result = argument

        # Save to memory
        self.memory.append({"user": user_input, "tool": tool_name, "response": str(result)})

        return {"tool_used": tool_name, "argument": argument, "result": result}

    def show_memory(self):
        """Display conversation history."""
        print("📝 Historia rozmowy:")
        for i, msg in enumerate(self.memory, 1):
            print(f"  {i}. [{msg['tool']}] {msg['user'][:50]}... → {msg['response'][:60]}...")


# Create agent and test memory
agent = ITAgent()

print("--- Rozmowa 1 ---")
r1 = agent.chat("Classify: Drukarka sieciowa nie odpowiada")
print(f"🔧 {r1['tool_used']}: {r1['result']}")

print("\n--- Rozmowa 2 ---")
r2 = agent.chat("Search the knowledge base for more info about this printer issue")
print(f"🔧 {r2['tool_used']}: {r2['result']}")

print("\n--- Rozmowa 3 ---")
r3 = agent.chat("What is the status of ticket T-002?")
print(f"🔧 {r3['tool_used']}: {r3['result']}")

print()
agent.show_memory()

---
## 🔹 Krok 4: Structured Output — JSON zamiast tekstu (~10 min)

W 2026 nie prosimy modelu o wolny tekst. Definiujemy **schemat JSON** i model go wypelnia.

### 📖 Wytlumaczenie:
To jest kluczowa roznica miedzy prompt engineering (2023) a context engineering (2026). Zamiast: "Odpowiedz jednym slowem", definiujemy schemat: `{category, confidence, reasoning, next_step}`. Model wypelnia strukture, kod ja parsuje. Zero niespodzianek, zero parsowania tekstu.

### 💡 Cwiczenie:
Dodaj pole `priority` (P1-P4) do schematu i sprawdz, czy model sensownie przypisuje priorytety.

In [ ]:
# === Structured output: model returns JSON, not free text ===

STRUCTURED_PROMPT = """Classify this IT ticket. Return ONLY valid JSON in this exact format:
{"category": "Hardware|Network|Software|Account",
 "confidence": 0.0 to 1.0,
 "reasoning": "one sentence why",
 "next_step": "recommended action"}

Ticket: "{ticket}"

JSON:"""


def classify_structured(ticket_text):
    """Classify a ticket and return structured JSON."""
    prompt = STRUCTURED_PROMPT.format(ticket=ticket_text)
    response = call_llm(prompt, system_prompt="You are a JSON-only responder. Never add text outside JSON.")

    try:
        start = response.index("{")
        end = response.rindex("}") + 1
        return json.loads(response[start:end])
    except (ValueError, json.JSONDecodeError):
        return {"error": "Failed to parse JSON", "raw": response}


# Test on ambiguous tickets
test_tickets = [
    "Drukarka sieciowa nie odpowiada",
    "System ERP nie pozwala na zmiane hasla",
    "Monitor migocze i komputer sie wylacza",
    "Brak dostepu do folderu sieciowego po zmianie hasla",
]

print("📋 Structured classification results:\n")
for ticket in test_tickets:
    result = classify_structured(ticket)
    print(f"📌 \"{ticket}\"")
    print(f"   {json.dumps(result, ensure_ascii=False, indent=2)}")
    print()

---
## 🔹 Krok 5: Prawdziwe dane — agent pracuje na CSV (~15 min)

Podlaczamy agenta do prawdziwych danych: `large_tickets.csv` z naszego wczesniejszego cwiczenia.

### 📖 Wytlumaczenie:
Prawdziwy agent nie dziala w prozni. Ma dostep do baz danych, plikow, API. Tu dodajemy narzedzie `query_tickets`, ktore odpowiada na pytania o dane. Agent decyduje, kiedy go uzyc — nie my.

### 💡 Cwiczenie:
Dodaj narzedzie `add_ticket(text, category)` ktore dodaje nowy wiersz do DataFrame.

In [ ]:
# Load real data
df = pd.read_csv('large_tickets.csv')
print(f"📊 Zaladowano {len(df)} zgloszen")
print(f"📂 Kategorie: {df['label'].value_counts().to_dict()}")

In [ ]:
# === New tool: query real ticket data ===

def query_tickets(question):
    """Answer questions about the ticket database."""
    q = question.lower()

    if "count" in q or "how many" in q or "ile" in q:
        if any(cat.lower() in q for cat in df['label'].unique()):
            for cat in df['label'].unique():
                if cat.lower() in q:
                    count = len(df[df['label'] == cat])
                    return f"{cat}: {count} tickets"
        return f"Total tickets: {len(df)}. Breakdown: {df['label'].value_counts().to_dict()}"

    elif "most common" in q or "top" in q or "popular" in q or "najczestsze" in q:
        top = df['text'].value_counts().head(5)
        return "Top 5 most common tickets:\n" + "\n".join(f"  {i+1}. {t} ({c}x)" for i, (t, c) in enumerate(top.items()))

    elif "sample" in q or "example" in q or "przyklad" in q:
        samples = df.sample(3)
        return "Random samples:\n" + "\n".join(f"  - [{r['label']}] {r['text']}" for _, r in samples.iterrows())

    else:
        return f"Database has {len(df)} tickets across {df['label'].nunique()} categories: {list(df['label'].unique())}"


# Add to tool registry
TOOLS["query_tickets"] = query_tickets

# Update agent system prompt
AGENT_SYSTEM_PROMPT_V2 = AGENT_SYSTEM_PROMPT + """
4. query_tickets(question) — query the IT ticket database (count, most common, samples)
"""

print("✅ Narzedzie query_tickets dodane!")
print(f"   Dostepne narzedzia: {list(TOOLS.keys())}")

In [ ]:
# === Full agent v2: with data access ===

agent_v2 = ITAgent()
agent_v2.system_prompt = AGENT_SYSTEM_PROMPT_V2

# Conversation with data
queries = [
    "How many tickets do we have in total?",
    "What are the most common ticket types?",
    "How many Network tickets are there?",
    "Classify this new ticket: Laptop sie nie wlacza po aktualizacji BIOS",
    "Search the knowledge base for laptop issues",
]

for q in queries:
    print(f"\n{'='*60}")
    print(f"👤 {q}")
    result = agent_v2.chat(q)
    print(f"🤖 [{result['tool_used']}] {result['result']}")

print(f"\n{'='*60}")
agent_v2.show_memory()

---
## 📊 Podsumowanie: Anatomia agenta AI

### 📖 Wytlumaczenie:
Wlasnie zbudowaliscie agenta AI od zera. Zobaczmy co ma w srodku:

```
┌─────────────────────────────────────┐
│           AGENT IT HELPDESK         │
├─────────────────────────────────────┤
│ 🧠 Mozg:     LLM (Gemma 3 4B)      │
│ 🔧 Narzedzia: classify, search_kb,  │
│              get_status, query_data  │
│ 📝 Pamiec:   lista wiadomosci       │
│ 📋 Output:   strukturalny JSON      │
│ 📊 Dane:     large_tickets.csv      │
└─────────────────────────────────────┘
```

| Komponent | Krok | Co robi |
|-----------|------|---------|
| LLM call | 1 | Bazowe wywolanie modelu |
| Tools | 2 | Agent decyduje, ktore narzedzie uzyc |
| Memory | 3 | Pamięta kontekst rozmowy |
| Structured output | 4 | Zwraca JSON, nie tekst |
| Data access | 5 | Pracuje na prawdziwych danych |

### 💡 Cwiczenie koncowe:
Rozszerz agenta o nowe narzedzie: `escalate_ticket(ticket_id, reason)` ktore oznacza zgloszenie jako pilne. Dodaj je do rejestru i przetestuj.